In [3]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model('human_detection_base.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

DATA_DIR = '../data/human-and-non-human/training_set'
IMG_SIZE = 96

def representative_data_gen():
    dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=1,
        color_mode="grayscale",
        shuffle=True
    )
    for images, labels in dataset.take(100):
        processed_image = (images.numpy() / 255.0).astype(np.float32)
        yield [processed_image]

converter.representative_dataset = representative_data_gen

# نکته کلیدی: فقط وزن‌های داخلی شبکه ۸-بیتی می‌شوند
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# *** خطوط مربوط به int8 کردن ورودی و خروجی را پاک کردیم ***

print("در حال تبدیل مدل (با ورودی و خروجی اعشاری)...")
tflite_quant_model = converter.convert()

with open('model.tflite', 'wb') as f:
    f.write(tflite_quant_model)
    
print("مدل با موفقیت ذخیره شد.")

در حال تبدیل مدل (با ورودی و خروجی اعشاری)...
INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmptabwe0d1\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmptabwe0d1\assets


Saved artifact at 'C:\Users\User\AppData\Local\Temp\tmptabwe0d1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2119305738000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305736848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305737232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305739728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305738192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305737424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305736464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305735504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305735888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305736080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2119305

c:\Users\User\anaconda3\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Found 8017 files belonging to 2 classes.
مدل با موفقیت ذخیره شد.


In [5]:
import os
# ذخیره فایل TFLite
tflite_model_path = 'model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)

# محاسبه و چاپ حجم مدل‌ها
base_model_size = os.path.getsize('human_detection_base.keras') / 1024
tflite_model_size = os.path.getsize(tflite_model_path) / 1024

print(f"حجم مدل اولیه: {base_model_size:.2f} KB")
print(f"حجم مدل کوانتیزه‌شده: {tflite_model_size:.2f} KB")
print(f"میزان فشرده‌سازی: {(base_model_size / tflite_model_size):.1f} برابر کوچکتر!")

حجم مدل اولیه: 553.52 KB
حجم مدل کوانتیزه‌شده: 48.53 KB
میزان فشرده‌سازی: 11.4 برابر کوچکتر!
